# Q01 — What Is the Difference Between a Pandas Series and a DataFrame?

**Python for Data Science — Company Interview Question Bank**

| Field | Detail |
|---|---|
| Question ID | `PANDAS-01-Q001` |
| Primary category | 01 — Pandas Fundamentals and Data Structures |
| Reported company | Legal & General |
| Reported position | Data Scientist |
| Question type | Candidate-reported; wording and attribution not independently reverified |
| Original difficulty | Fundamental |
| Extended depth | Advanced–Expert |
| Topics | Series, DataFrame, Index, dtype, alignment, memory, copying |
| Source | [Glassdoor candidate report](https://www.glassdoor.com/Interview/-Tell-me-about-a-time-where-you-have-worked-with-group-Tell-me-about-a-time-where-you-took-some-key-decisions-which-imp-QTN_9026922.htm) |

> **Source note:** The company association comes from a publicly attributed candidate report previously identified for this question. The exact wording, interview date, and attribution could not be independently reconfirmed here. The code, explanations, and follow-up questions below are original educational extensions, **not additional reported company questions**.

## 1. Interview Question

**What is the difference between a Pandas Series and a DataFrame?**

A strong response should cover **dimensionality**, **axes**, **dtype behavior**, **label alignment**, and why physical storage must not be inferred solely from a conceptual description.

## 2. Interview-Ready Answer

A **Pandas Series** is a one-dimensional labeled array with one logical dtype. A **Pandas DataFrame** is a two-dimensional labeled table with row and column axes; individual columns may have different dtypes.

| Property | Series | DataFrame |
|---|---|---|
| Dimensions | 1 (`ndim == 1`) | 2 (`ndim == 2`) |
| Shape | `(n,)` | `(n, m)` |
| Axes | Index | Row index and column index |
| Dtypes | One logical dtype | One dtype per column, potentially heterogeneous |
| Selection | Scalar or Series, depending on indexing | Scalar, Series, or DataFrame, depending on indexing |
| Common use | Single labeled feature or time series | Multi-column analytical dataset |

A DataFrame can be **conceptually** viewed as multiple aligned Series, but this is **not a guarantee of physical memory layout**. Both are index-aware and may automatically align data by labels rather than position.

## 3. Setup

The following code cells are executable and designed to work with common Pandas 2.x and 3.x configurations. Some display details (especially string dtypes) depend on version and configuration.

In [1]:
import sys
import numpy as np
import pandas as pd

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.12.10
Pandas: 2.3.3
NumPy: 2.2.6


## 4. Basic Implementation — Structure, Shape, and Dtypes

Compare a single labeled vector with a table containing numeric and string columns. Check properties with assertions rather than depending only on printed formatting.

In [2]:
revenue = pd.Series([100, 200, 300], index=["A", "B", "C"], name="Revenue")

sales = pd.DataFrame(
    {
        "Revenue": [100, 200, 300],
        "Cost": [60, 120, 180],
        "Region": ["East", "West", "North"],
    },
    index=["A", "B", "C"],
)

print("SERIES")
print(revenue)
print("\nDATAFRAME")
print(sales)
print("\nSeries shape / ndim / dtype:", revenue.shape, revenue.ndim, revenue.dtype)
print("DataFrame shape / ndim:\n", sales.shape, sales.ndim)
print("DataFrame dtypes:\n", sales.dtypes)

assert isinstance(revenue, pd.Series)
assert isinstance(sales, pd.DataFrame)
assert revenue.shape == (3,) and revenue.ndim == 1
assert sales.shape == (3, 3) and sales.ndim == 2
assert pd.api.types.is_integer_dtype(sales["Revenue"].dtype)
assert pd.api.types.is_integer_dtype(sales["Cost"].dtype)

SERIES
A    100
B    200
C    300
Name: Revenue, dtype: int64

DATAFRAME
   Revenue  Cost Region
A      100    60   East
B      200   120   West
C      300   180  North

Series shape / ndim / dtype: (3,) 1 int64
DataFrame shape / ndim:
 (3, 3) 2
DataFrame dtypes:
 Revenue     int64
Cost        int64
Region     object
dtype: object


## 5. Column Selection — Series vs. DataFrame

- `df["column"]` returns a Series when the column label uniquely identifies one column.
- `df[["column"]]` returns a DataFrame.

**Caveat:** Duplicate column labels can change single-label selection behavior, so these examples deliberately use unique labels.

In [3]:
one_column_series = sales["Revenue"]
one_column_frame = sales[["Revenue"]]

print("Series selection:", type(one_column_series).__name__, one_column_series.shape)
print("DataFrame selection:", type(one_column_frame).__name__, one_column_frame.shape)

assert isinstance(one_column_series, pd.Series)
assert isinstance(one_column_frame, pd.DataFrame)
assert one_column_series.shape == (3,)
assert one_column_frame.shape == (3, 1)

Series selection: Series (3,)
DataFrame selection: DataFrame (3, 1)


## 6. Label Alignment — A Critical Pandas Behavior

For Series with unique labels, arithmetic aligns matching labels before combining values, rather than pairing elements based solely on their physical position.

If $a_i$ and $b_i$ are values indexed by the same label $i$, then the aligned result is

$$
(a+b)_i=a_i+b_i
$$

where a label missing from either input normally produces a missing result for ordinary addition. Handling duplicate index labels requires additional care.

In [4]:
left = pd.Series([10, 20, 30], index=["A", "B", "C"])
right = pd.Series([100, 200, 300], index=["C", "A", "D"])

aligned_sum = left + right
print("Left:\n", left)
print("\nRight:\n", right)
print("\nAligned sum:\n", aligned_sum)

assert aligned_sum.loc["A"] == 210
assert aligned_sum.loc["C"] == 130
assert pd.isna(aligned_sum.loc["B"])
assert pd.isna(aligned_sum.loc["D"])

# Contrast: NumPy addition is positional, not label-aligned.
positional_sum = left.to_numpy() + right.to_numpy()
print("\nPositional NumPy sum:", positional_sum)
assert positional_sum.tolist() == [110, 220, 330]

Left:
 A    10
B    20
C    30
dtype: int64

Right:
 C    100
A    200
D    300
dtype: int64

Aligned sum:
 A    210.0
B      NaN
C    130.0
D      NaN
dtype: float64

Positional NumPy sum: [110 220 330]


## 7. Storage and Copy Semantics

A Series may be backed by NumPy data or a Pandas extension array. DataFrames may use multiple buffers or internal blocks. Their conceptual shape alone does not specify allocation or contiguity.

An **explicit numeric copy** avoids mutating the source through the copied Series. Whether intermediate objects share storage, and whether writes propagate, depends on the operation and Pandas' Copy-on-Write semantics. Pandas 3.x uses Copy-on-Write by default.

**Important:** `.copy(deep=True)` does not recursively clone arbitrary mutable Python objects stored in an `object` column.

In [5]:
prices_df = pd.DataFrame({"Price": [100.0, 200.0, 300.0], "Volume": [1000, 2000, 3000]})
prices_copy = prices_df["Price"].copy()
prices_copy.iloc[0] = 999.0

print("Original DataFrame:\n", prices_df)
print("\nIndependent Series:\n", prices_copy)

assert prices_df.loc[0, "Price"] == 100.0
assert prices_copy.iloc[0] == 999.0

Original DataFrame:
    Price  Volume
0  100.0    1000
1  200.0    2000
2  300.0    3000

Independent Series:
 0    999.0
1    200.0
2    300.0
Name: Price, dtype: float64


### 7.1 A Subtle Deep-Copy Limitation

Even with `deep=True`, nested Python objects can remain shared. This example demonstrates why "deep" in Pandas does not imply recursive Python `copy.deepcopy` behavior.

In [6]:
nested = pd.Series([[1, 2], [3, 4]], dtype=object)
nested_copy = nested.copy(deep=True)

# The Series containers are distinct, but the nested Python list can be shared.
print("Same Series object?", nested is nested_copy)
print("Same nested list object?", nested.iloc[0] is nested_copy.iloc[0])

assert nested is not nested_copy
assert nested.iloc[0] is nested_copy.iloc[0]

# Avoid mutating either nested object in order to keep subsequent cells independent.

Same Series object? False
Same nested list object? True


## 8. Complexity and Memory Analysis

Let $n$ be the row count, $m$ the column count, and $s$ the bytes per element for homogeneous fixed-width data.

| Operation | Typical cost | Qualification |
|---|---|---|
| Read `ndim` or `shape` | $O(1)$ | Metadata-only access |
| Build a Series from materialized values | $O(n)$ | May avoid copying with some input pathways |
| Build a fully materialized $n\times m$ table | $O(nm)$ | Construction overhead depends on input/dtypes |
| Elementwise arithmetic on already-aligned Series | $O(n)$ | Index alignment may add work |
| Copy numeric Series values | $O(n)$ | Copies fixed-width data buffer |
| Select a column by label | Implementation-dependent | Index lookup and object construction costs vary |

Approximate **data-buffer** storage:

$$
M_{\mathrm{Series}}\approx ns,\qquad
M_{\mathrm{DataFrame}}\approx nms.
$$

These estimates omit indices, labels, metadata, object references, variable-length values, and temporary allocations. `object` and extension dtypes can require substantially different memory models.

## 9. Common Interview Pitfalls

1. **Series is not just a list:** it has an index, a dtype, and alignment semantics.
2. **A DataFrame is not necessarily a homogeneous matrix:** column dtypes can differ.
3. **`.loc` is label-based, `.iloc` is position-based:** confusing them can silently select the wrong observations.
4. **Series arithmetic does not necessarily use positional pairing:** indexes matter.
5. **A column selection is not guaranteed to be an independent copy:** avoid assumptions about buffer ownership.
6. **Copy-on-Write affects assignment behavior:** inspect the Pandas version and semantics.
7. **Nonunique labels exist:** duplicate row/column labels complicate selection and alignment.
8. **`deep=True` is not recursively deep for arbitrary Python objects.**

## 10. Advanced Follow-up Questions — Educational Extensions

**F1. Why can two Series with identical values but different labels return different results when added?**  
Because Pandas aligns by labels; unmatched labels introduce missing observations.

**F2. Can a Series contain different Python object types despite having one dtype?**  
Yes. An `object` Series can store heterogeneous Python objects while exposing the single logical dtype `object`.

**F3. Why can `DataFrame.to_numpy()` be expensive?**  
Heterogeneous columns may require conversion to a common dtype, allocation, and copying.

**F4. Does `df["Price"]` always return a Series?**  
Not when column labels are duplicated: selecting a duplicated label can return a DataFrame.

**F5. How can label alignment cause problems with financial observations?**  
Bad timestamps, inconsistent calendars, or unintended outer alignment may introduce missing values or misleading comparisons.

**F6. Does a Pandas deep copy recursively duplicate nested objects?**  
No; mutable Python objects stored inside object-dtype arrays may still be referenced by both containers.

In [7]:
# Follow-up F4: demonstrate duplicated column labels
with_duplicate_columns = pd.DataFrame([[10, 20]], columns=["Price", "Price"])
selected = with_duplicate_columns["Price"]

print("Selected object type:", type(selected).__name__)
print(selected)
assert isinstance(selected, pd.DataFrame)

Selected object type: DataFrame
   Price  Price
0     10     20


## 11. Analytical Interpretation

The practical distinction is not only dimensionality. The choice between Series and DataFrame affects index alignment, dtype representation, downstream transformation APIs, and the potential for memory allocation.

- **Series** commonly models one measured variable, return series, label vector, or feature.
- **DataFrame** commonly models a set of related variables, with both row and column semantics.
- **Index discipline** matters for time series, financial datasets, and joined business metrics.
- **Copy awareness** matters when feature engineering, caching, and slicing large datasets.

A strong candidate moves beyond "1D versus 2D" to explain which assumptions remain valid under missing data, duplicated labels, mixed dtypes, and Copy-on-Write.

## 12. Concise Interview-Ready Summary

> A Series is a one-dimensional labeled array with one logical dtype. A DataFrame is a two-dimensional labeled table whose columns may have different dtypes. Both use labeled axes, so many operations align by index rather than by position. Selecting one unique column usually gives a Series, whereas selecting a list of columns gives a DataFrame. Their memory and copy behavior should be understood separately from their conceptual structure.

**Completion check:** Explain the difference in under one minute; run the examples; predict the alignment results without executing them; and justify the distinction between conceptual structure and physical storage.